# Question 3: TF-IDF Representation & Sentence Similarity

**Course:** Natural Language Processing (AI357)  
**Institution:** Sardar Vallabhbhai National Institute of Technology (SVNIT), Surat  

---

### Problem Statement:
Consider the following sentences in the toy dataset:
1. `cat dog animal`
2. `cat cat pet`
3. `dog pet animal`
4. `cat dog pet`

**Tasks:**
- Represent each word in terms of a TF-IDF score.
- Use raw counts for TF and unsmoothed IDF while computing TF-IDF.
- Which two sentences are closest to each other? Show all computations.
- Don't add any start and end sentence markers.
- [Hint: Use TF-IDF vectors to represent each sentence.]  
**Marks:** $[4 + 2 = 6]$


## 1. Mathematical Formulas

1. **Raw Term Frequency (TF):**
   $$\text{TF}(w, d) = \text{count of word } w \text{ in sentence } d$$

2. **Unsmoothed Inverse Document Frequency (IDF):**
   $$\text{IDF}(w) = \ln\left(\frac{N}{\text{DF}(w)}\right)$$
   where $N = 4$ is the total number of sentences, and $\text{DF}(w)$ is the document frequency (number of sentences containing $w$).

3. **TF-IDF Weight:**
   $$\text{TF-IDF}(w, d) = \text{TF}(w, d) \times \text{IDF}(w)$$

4. **Cosine Similarity:**
   $$\text{CosineSim}(\mathbf{d}_i, \mathbf{d}_j) = \frac{\mathbf{d}_i \cdot \mathbf{d}_j}{\|\mathbf{d}_i\| \|\mathbf{d}_j\|}$$


In [1]:
import math
import pandas as pd
from collections import Counter

sentences = [
    "cat dog animal",   # S1
    "cat cat pet",      # S2
    "dog pet animal",   # S3
    "cat dog pet"       # S4
]
sentence_labels = ["S1", "S2", "S3", "S4"]
num_sentences = len(sentences)

# Vocabulary in alphabetical order
vocabulary = sorted(list(set(word for sentence in sentences for word in sentence.split())))
print(f"Total Sentences (N): {num_sentences}")
print(f"Vocabulary ({len(vocabulary)} words): {vocabulary}")


Total Sentences (N): 4
Vocabulary (4 words): ['animal', 'cat', 'dog', 'pet']


In [2]:
# Step 1: Raw Term Frequency (TF)
tf_matrix = []
for sentence in sentences:
    counts = Counter(sentence.split())
    tf_matrix.append({word: counts[word] for word in vocabulary})

df_tf = pd.DataFrame(tf_matrix, index=sentence_labels)
print("=== Raw Term Frequency (TF) Matrix ===")
df_tf


=== Raw Term Frequency (TF) Matrix ===


,animal,cat,dog,pet
S1,1,1,1,0
S2,0,2,0,1
S3,1,0,1,1
S4,0,1,1,1


In [3]:
# Step 2: Document Frequency (DF) and Unsmoothed IDF
df_counts = {}
for word in vocabulary:
    df_counts[word] = sum(1 for row in tf_matrix if row[word] > 0)

idf_values = {}
for word in vocabulary:
    idf_values[word] = math.log(num_sentences / df_counts[word])  # natural logarithm

df_idf = pd.DataFrame({
    "Word": vocabulary,
    "DF (Document Frequency)": [df_counts[w] for w in vocabulary],
    "N / DF": [f"{num_sentences}/{df_counts[w]}" for w in vocabulary],
    "Unsmoothed IDF = ln(N/DF)": [idf_values[w] for w in vocabulary]
})
print("=== Document Frequency & Unsmoothed IDF Table ===")
df_idf


=== Document Frequency & Unsmoothed IDF Table ===


,Word,DF (Document Frequency),N / DF,Unsmoothed IDF = ln(N/DF)
0,animal,2,4/2,0.693147
1,cat,3,4/3,0.287682
2,dog,3,4/3,0.287682
3,pet,3,4/3,0.287682


In [4]:
# Step 3: Compute TF-IDF Matrix
tfidf_vectors = []
for row in tf_matrix:
    tfidf_row = {word: row[word] * idf_values[word] for word in vocabulary}
    tfidf_vectors.append(tfidf_row)

df_tfidf = pd.DataFrame(tfidf_vectors, index=sentence_labels)
print("=== TF-IDF Matrix (Raw TF * Unsmoothed IDF) ===")
df_tfidf


=== TF-IDF Matrix (Raw TF * Unsmoothed IDF) ===


,animal,cat,dog,pet
S1,0.693147,0.287682,0.287682,0.000000
S2,0.000000,0.575364,0.000000,0.287682
S3,0.693147,0.000000,0.287682,0.287682
S4,0.000000,0.287682,0.287682,0.287682


In [5]:
# Step 4: Compute Pairwise Cosine Similarity
def calculate_cosine_similarity(vec_a, vec_b):
    dot_product = sum(vec_a[w] * vec_b[w] for w in vocabulary)
    norm_a = math.sqrt(sum(vec_a[w] ** 2 for w in vocabulary))
    norm_b = math.sqrt(sum(vec_b[w] ** 2 for w in vocabulary))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

sim_matrix = []
pair_details = []

for i in range(num_sentences):
    row_sim = []
    for j in range(num_sentences):
        sim = calculate_cosine_similarity(tfidf_vectors[i], tfidf_vectors[j])
        row_sim.append(sim)
        if i < j:
            pair_details.append({
                "Pair": f"({sentence_labels[i]}, {sentence_labels[j]})",
                "Sentence 1": sentences[i],
                "Sentence 2": sentences[j],
                "Cosine Similarity": sim
            })
    sim_matrix.append(row_sim)

df_sim_matrix = pd.DataFrame(sim_matrix, index=sentence_labels, columns=sentence_labels)
df_ranked_pairs = pd.DataFrame(pair_details).sort_values("Cosine Similarity", ascending=False)

print("=== Cosine Similarity Matrix ===")
display(df_sim_matrix)

print("\n=== Ranked Pairs by Cosine Similarity ===")
display(df_ranked_pairs)


=== Cosine Similarity Matrix ===


,S1,S2,S3,S4
S1,1.000000,0.320147,0.871882,0.413308
S2,0.320147,1.000000,0.160074,0.774597
S3,0.871882,0.160074,1.000000,0.413308
S4,0.413308,0.774597,0.413308,1.000000



=== Ranked Pairs by Cosine Similarity ===


,Pair,Sentence 1,Sentence 2,Cosine Similarity
1,"(S1, S3)",cat dog animal,dog pet animal,0.871882
4,"(S2, S4)",cat cat pet,cat dog pet,0.774597
5,"(S3, S4)",dog pet animal,cat dog pet,0.413308
2,"(S1, S4)",cat dog animal,cat dog pet,0.413308
0,"(S1, S2)",cat dog animal,cat cat pet,0.320147
3,"(S2, S3)",cat cat pet,dog pet animal,0.160074


## Summary of Answers

1. **TF-IDF Representation [4 Marks]:**
   - Vocabulary: `['animal', 'cat', 'dog', 'pet']`
   - IDF values:
     - $\text{IDF}(\text{animal}) = \ln(4/2) = \ln(2) \approx 0.693147$
     - $\text{IDF}(\text{cat}) = \ln(4/3) \approx 0.287682$
     - $\text{IDF}(\text{dog}) = \ln(4/3) \approx 0.287682$
     - $\text{IDF}(\text{pet}) = \ln(4/3) \approx 0.287682$

2. **Closest Sentences [2 Marks]:**
   - **Sentence S1 (`cat dog animal`) and Sentence S3 (`dog pet animal`) are the closest to each other.**
   - **Cosine Similarity:** **`0.871882`**
   - **Reason:** Both sentences share `'animal'` (which has the highest IDF weight since it is the rarest term) and `'dog'`.
